# Part 1 — PASTIS and labels

**Goal:** the Drive holds
- `processed/`: `meta.csv`, `labels.npy`, `instances.npy`, `parcel_ids.npy`, `splits.json`, `dates_s2.json`, `class_pixel_counts.csv`, `selection_seed42.json`;
- `pastis_subset/`: about 45 patches (the 25 of the RF pilot + 20 QA patches), so that the next parts no longer need the 28.8 GB zip.

**Runtime:** Runtime → Change runtime type → **CPU**. **Time:** 30–70 min, mostly the PASTIS download. **Drive space for this part:** about 1–1.5 GB (estimate), plus 28.8 GB if the zip is kept.

## 1. Runtime and Drive
**Check:** Python ≥ 3.12 (required by geotessera) and at least 40 GB free in `/content`.

In [ ]:
import sys, shutil, platform
print("Python", platform.python_version())
assert sys.version_info >= (3, 12), "geotessera needs Python >= 3.12"
print(f"free space in /content: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!df -h /content/drive | tail -1

**Configuration.** If `pj_vmi` is a folder shared with you, add it to your Drive first (drive.google.com → right click → Organize → Add shortcut → My Drive); it then appears as `/content/drive/MyDrive/pj_vmi` (or the shortcut name). Files you create in a shared folder count against **your** storage quota.

Set `BRANCH` to the branch that holds the code.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")

ZIP = Path("/content/PASTIS.zip")
ZIP_DRIVE = DRIVE / "raw" / "PASTIS.zip"
KEEP_ZIP_ON_DRIVE = False          # True if the Drive has >= 30 GB free: no new download next time
PROC = Path("/content/processed")   # work on the local disk (fast), then copy to the Drive
PROC_DRIVE = DRIVE / "processed"
SUBSET = DRIVE / "pastis_subset"

for p in [PROC, PROC_DRIVE]:
    p.mkdir(parents=True, exist_ok=True)

## 2. Code
Clones the repository the first time, pulls afterwards, and prints the commit in use: write it down with the results.

**Common errors:** `Remote branch ... not found` → wrong `BRANCH`; missing files later → some files listed at the top were not pushed.

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -c "import geotessera, geopandas; print('geotessera', geotessera.__version__, '| geopandas', geopandas.__version__)"

## 3. PASTIS download
Source: Zenodo record 5012942, `PASTIS.zip`, 28,760,245,504 bytes, md5 `cfc441bf18137ff0bbf4fad58828fb98`, CC-BY 4.0. The zip is **not extracted**: the scripts read it directly (extracted, it takes 39.5 GB).

**Common errors:** slow or interrupted Zenodo download → run the cell again (`wget -c` resumes). If the runtime is reset, `/content` is lost and the zip must be downloaded again, unless it was kept on the Drive.

In [ ]:
URL = "https://zenodo.org/api/records/5012942/files/PASTIS.zip/content"
SIZE, MD5 = 28_760_245_504, "cfc441bf18137ff0bbf4fad58828fb98"

if ZIP.exists() and ZIP.stat().st_size == SIZE:
    print("PASTIS.zip already complete")
elif ZIP_DRIVE.exists():
    print("restoring from the Drive..."); shutil.copy(ZIP_DRIVE, ZIP)
else:
    get_ipython().system(f'wget -c -q --show-progress -O "{ZIP}" "{URL}"')
print("size ok:", ZIP.stat().st_size == SIZE)

In [ ]:
out = !md5sum "{ZIP}"
print(out[0])
assert out[0].split()[0] == MD5, "wrong md5: delete the file and run the download cell again"
if KEEP_ZIP_ON_DRIVE and not ZIP_DRIVE.exists():
    ZIP_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(ZIP, ZIP_DRIVE)   # about 10-30 min

## 4. meta.csv and labels
The PASTIS pixel grid is the UTM grid of each Sentinel-2 tile, not Lambert-93. The top-left corner of each patch is snapped to the nearest multiple of 10 m; the raw corner (used by the bilinear alignment of the RF pilot) is kept as `x0_raw` / `y1_raw`. The zip has 2,468 S2 files but only 2,433 labelled patches: the 35 extra S2 files are not in `metadata.geojson` and are ignored.

**Check**:
- 2,433 patches;
- folds {1: 487, 2: 494, 3: 474, 4: 482, 5: 496};
- tiles {t30uxv: 531, t31tfj: 623, t31tfm: 723, t32ulu: 556};
- S2 `int16`, every patch with as many dates as in the metadata;
- no `WARNING ... footprints` line;
- constant snap correction within each tile: t30uxv (−2.36, −0.62), t31tfj (0.90, −3.51), t31tfm (1.84, 0.05), t32ulu (−2.27, −0.49) m.

In [ ]:
!python scripts/build_meta_labels.py --pastis "{ZIP}" --out "{PROC}"

In [ ]:
meta = pd.read_csv(PROC / "meta.csv", index_col="idx")
labels = np.load(PROC / "labels.npy", mmap_mode="r")
assert len(meta) == 2433 and labels.shape == (2433, 128, 128)
print("footprint_ok:", meta.footprint_ok.value_counts().to_dict())
print(meta.groupby("TILE")[["snap_dx", "snap_dy"]].agg(["min", "max"]).round(2))
# pixels per class and fold: compare with Figure 3 of the PASTIS documentation
display(pd.read_csv(PROC / "class_pixel_counts.csv", index_col=0).T)

**The 25 patches of the RF pilot.** This list must match `selection.json` written by `baselines/rf/prepare_batch.py`. If that file is on the Drive at `pj_vmi/rf_pilot/batch_2019/selection.json`, the cell compares them; otherwise compare by eye.

In [ ]:
sel = json.loads((PROC / "selection_seed42.json").read_text())
mine = [(p["patch_id"], p["fold"]) for p in sel["patches"]]
print(mine)
rf_file = DRIVE / "rf_pilot" / "batch_2019" / "selection.json"
if rf_file.exists():
    rf = [(p["patch_id"], p["fold"]) for p in json.loads(rf_file.read_text())["patches"]]
    print("same as the RF pilot selection.json:", mine == rf)

In [ ]:
small = ["meta.csv", "labels.npy", "instances.npy", "parcel_ids.npy", "splits.json", "dates_s2.json",
         "class_pixel_counts.csv", "selection_seed42.json"]
for f in small:
    shutil.copy(PROC / f, PROC_DRIVE / f)
    print(f"{f:24s} {(PROC_DRIVE / f).stat().st_size / 1e6:8.1f} MB")

## 5. PASTIS subset for the next parts
Copies from the zip to the Drive the files of about 45 patches (`TARGET`, `ParcelIDs`, `INSTANCES`, `S2`, plus `metadata.geojson`), in the PASTIS layout, so the RF pilot scripts and parts 2–4 read it like the full dataset.

**Check:** the last line gives the number of patches (`selection_seed42: 25, qa: 20`) and the size (about 0.5–1 GB).

In [ ]:
!python scripts/extract_subset.py --pastis "{ZIP}" --processed "{PROC}" --out "{SUBSET}"